# ReplicaLab — GRPO Fine-Tuning (Qwen3.5-0.8B)

Trains `Qwen/Qwen3.5-0.8B` with GRPO on the ReplicaLab negotiation environment.
The Scientist agent learns to propose protocols that satisfy Lab Manager constraints.

**Environment:** `https://ayushozha-replicalab.hf.space`  
**Reward signal:** per-step protocol delta + final agreement bonus  
**Method:** GRPO via `trl` + Unsloth  

---
Run cells top-to-bottom on an H100 / A100. Takes ~30–60 min for 200 steps.

In [1]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
# Skip if packages already installed (e.g. pre-installed on the H100 runtime).
import subprocess, sys, importlib

def _ensure(pkg, import_name=None):
    name = import_name or pkg.split(">=")[0].split("[")[0]
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("trl>=0.12", "trl")
_ensure("pydantic>=2.7,<3.0", "pydantic")
_ensure("httpx>=0.27", "httpx")
_ensure("datasets>=2.19", "datasets")
_ensure("accelerate>=0.30", "accelerate")
_ensure("papermill", "papermill")
_ensure("bitsandbytes", "bitsandbytes")

try:
    import unsloth
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth_zoo"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])

print("All dependencies ready.")

/tmp/ipykernel_12102/2495779305.py:21: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


All dependencies ready.


In [2]:
# ── 1. Config ─────────────────────────────────────────────────────────────────
import os

# Model
MODEL_ID         = "unsloth/Qwen3.5-0.8B"   # ~800M params, fits in 24 GB VRAM with 4-bit LoRA
MAX_SEQ_LEN      = 2048
LORA_RANK        = 16
LORA_ALPHA       = 32

# Training
GRPO_STEPS       = 200          # total gradient steps
BATCH_SIZE       = 4            # prompts per step
NUM_GENERATIONS  = 4            # completions per prompt (GRPO sample count)
LR               = 5e-5
MAX_NEW_TOKENS   = 512          # max tokens for each generated action

# Environment
ENV_BASE_URL     = os.getenv("REPLICALAB_URL", "https://ayushozha-replicalab.hf.space")
SCENARIOS        = ["math_reasoning", "ml_benchmark", "finance_trading"]
DIFFICULTIES     = ["easy", "medium"]
EVAL_SEEDS       = list(range(10))   # fixed held-out seeds for before/after comparison

# Output
OUTPUT_DIR       = "./replicalab-qwen3.5-grpo"
HF_PUSH_REPO     = None   # set to "your-hf-username/replicalab-qwen-grpo" to push on completion

print(f"Model : {MODEL_ID}")
print(f"Env   : {ENV_BASE_URL}")
print(f"Steps : {GRPO_STEPS}")

Model : unsloth/Qwen3.5-0.8B
Env   : https://ayushozha-replicalab.hf.space
Steps : 200


In [3]:
# ── 2. Environment client ─────────────────────────────────────────────────────
import json, random, time
import httpx

class EnvClient:
    """Thin REST client for the ReplicaLab environment."""

    def __init__(self, base_url: str, timeout: float = 30.0):
        self._base = base_url.rstrip("/")
        self._http = httpx.Client(timeout=timeout)
        self._session_id: str | None = None

    def health(self) -> dict:
        return self._http.get(f"{self._base}/health").json()

    def reset(self, seed: int = 0, scenario: str = "math_reasoning",
              difficulty: str = "easy") -> dict:
        r = self._http.post(f"{self._base}/reset",
                            json={"seed": seed, "scenario": scenario,
                                  "difficulty": difficulty})
        r.raise_for_status()
        data = r.json()
        self._session_id = data["session_id"]
        return data

    def step(self, action: dict) -> dict:
        assert self._session_id, "Call reset() first"
        r = self._http.post(f"{self._base}/step",
                            json={"session_id": self._session_id, "action": action})
        r.raise_for_status()
        return r.json()

    def close(self):
        self._http.close()


# Smoke test
client = EnvClient(ENV_BASE_URL)
h = client.health()
print("Health:", h)
assert h["status"] == "ok", "Environment is not healthy!"
print("Environment is up and using:", h.get("env", "unknown"))

Health: {'status': 'ok', 'env': 'real', 'version': '0.1.0'}
Environment is up and using: real


In [4]:
# ── 3. Prompt / action schema ─────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a Scientist agent in a multi-agent negotiation environment.
Your goal is to propose an experimental protocol that the Lab Manager will accept.
The Lab Manager enforces budget, equipment, reagent, staffing, and schedule constraints.

On each turn you must output a single JSON object matching this schema:
{
  "action_type": "propose_protocol" | "revise_protocol" | "request_info" | "accept",
  "sample_size": <int >= 1 for propose/revise, 0 otherwise>,
  "controls": ["..."],
  "technique": "<method name>",
  "duration_days": <int>,
  "required_equipment": ["..."],
  "required_reagents": ["..."],
  "questions": [],
  "rationale": "<brief justification>"
}

Rules:
- For propose_protocol / revise_protocol: sample_size >= 1, technique and rationale required, questions must be empty.
- For request_info: questions must be non-empty, all other fields zeroed out.
- For accept: all fields zeroed / empty.
- Output ONLY the JSON object — no markdown, no explanation."""


def obs_to_prompt(obs: dict) -> str:
    """Format a ScientistObservation into a user-turn prompt."""
    sci = obs["scientist"]
    lab = obs["lab_manager"]

    history_lines = []
    for entry in sci.get("conversation_history", []):
        role = entry["role"].upper()
        history_lines.append(f"[{role}] {entry['message']}")
    history = "\n".join(history_lines) if history_lines else "(none yet)"

    return f"""\
=== PAPER ===
Title: {sci['paper_title']}
Hypothesis: {sci['paper_hypothesis']}
Method: {sci['paper_method']}
Key finding: {sci['paper_key_finding']}
Goal: {sci['experiment_goal']}

=== LAB CONSTRAINTS ===
Budget remaining: {lab['budget_remaining']} / {lab['budget_total']}
Equipment available: {', '.join(lab['equipment_available'])}
Reagents in stock: {', '.join(lab['reagents_in_stock'])}
Staff: {lab['staff_count']}
Time limit: {lab['time_limit_days']} days
Safety restrictions: {'; '.join(lab['safety_restrictions']) or 'none'}

=== ROUND {sci['round_number']} / {sci['max_rounds']} ===
Conversation so far:
{history}

Now output your action as a JSON object."""


def make_chat_messages(obs: dict) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": obs_to_prompt(obs)},
    ]


# Quick test
test_reset = client.reset(seed=0, scenario="math_reasoning", difficulty="easy")
print(obs_to_prompt(test_reset["observation"])[:600], "...")

=== PAPER ===
Title: Planning a proof of the Cauchy-Schwarz inequality
Hypothesis: A square-expansion argument gives the cleanest proof path.
Method: Outline the proof using one algebraic identity, one equality-case check, and reviewer notes.
Key finding: The proof is accepted only if every inequality step and equality case is justified.
Goal: Produce a proof-planning workflow for the Cauchy-Schwarz inequality for an undergraduate seminar handout.

=== LAB CONSTRAINTS ===
Budget remaining: 345.0 / 345.0
Equipment available: Structured proof notebook
Reagents in stock: Reference theorem library ...


In [5]:
# ── 4. Build training dataset ─────────────────────────────────────────────────
# We pre-collect (seed, scenario, difficulty) triplets and format the INITIAL
# observation as the GRPO prompt. During training the reward function re-runs
# the episode from the same seed so we always compare apples to apples.

from datasets import Dataset

NUM_TRAIN_PROMPTS = 80   # unique prompts in the training pool

rows = []
rng = random.Random(42)

for i in range(NUM_TRAIN_PROMPTS):
    seed = rng.randint(100, 9999)
    scenario  = rng.choice(SCENARIOS)
    difficulty = rng.choice(DIFFICULTIES)
    try:
        reset_data = client.reset(seed=seed, scenario=scenario, difficulty=difficulty)
        obs = reset_data["observation"]
        rows.append({
            "seed": seed,
            "scenario": scenario,
            "difficulty": difficulty,
            "prompt": make_chat_messages(obs),
        })
    except Exception as e:
        print(f"[skip] seed={seed} {scenario}/{difficulty}: {e}")

dataset = Dataset.from_list(rows)
print(f"Dataset: {len(dataset)} prompts")
print(dataset[0]["prompt"][1]["content"][:300])

Dataset: 80 prompts
=== PAPER ===
Title: Planning a proof of Jensen's inequality for convex quadratics
Hypothesis: A convexity-first outline is shorter than an expectation-expansion route.
Method: Use the convexity definition, midpoint intuition, and one numerical sanity check.
Key finding: The plan succeeds only if th


In [6]:
# ── 5. Action parser ──────────────────────────────────────────────────────────
import re

VALID_ACTION_KEYS = {
    "action_type", "sample_size", "controls", "technique",
    "duration_days", "required_equipment", "required_reagents",
    "questions", "rationale",
}

DEFAULT_ACTION = {
    "action_type": "propose_protocol",
    "sample_size": 3,
    "controls": ["baseline"],
    "technique": "standard replication",
    "duration_days": 5,
    "required_equipment": [],
    "required_reagents": [],
    "questions": [],
    "rationale": "Default protocol.",
}


def _extract_text(completion) -> str:
    """TRL GRPO may pass completions as strings, lists of strings, or lists of
    message dicts. Normalise to a plain string (assistant content only)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                # Only take assistant role content to avoid picking up prompt JSON
                if item.get("role") in (None, "assistant"):
                    parts.append(str(item.get("content", "")))
        return " ".join(parts)
    return str(completion)


def _clean_list_field(val) -> list:
    """Ensure list field has only non-empty strings."""
    if not isinstance(val, list):
        return []
    return [str(v).strip() for v in val if str(v).strip()]


def parse_action(completion) -> dict:
    """Extract and validate a ScientistAction dict from a model completion."""
    text = _extract_text(completion)
    text = re.sub(r"```(?:json)?\s*", "", text).strip()
    start = text.find("{")
    end   = text.rfind("}")
    if start == -1 or end == -1:
        return dict(DEFAULT_ACTION)
    try:
        raw = json.loads(text[start : end + 1])
    except Exception:
        return dict(DEFAULT_ACTION)

    # Keep only valid keys (extra="forbid" in ScientistAction rejects extras)
    obj = {k: raw[k] for k in VALID_ACTION_KEYS if k in raw}

    # Fill in defaults for missing keys
    for k, v in DEFAULT_ACTION.items():
        obj.setdefault(k, v)

    # Ensure list fields have no empty strings (validator rejects them)
    for key in ("controls", "required_equipment", "required_reagents", "questions"):
        obj[key] = _clean_list_field(obj[key])

    # Ensure controls is non-empty for propose/revise (use default if needed)
    if obj["action_type"] in ("propose_protocol", "revise_protocol"):
        if not obj["controls"]:
            obj["controls"] = ["baseline"]
        if not obj.get("technique", "").strip():
            obj["technique"] = DEFAULT_ACTION["technique"]
        if not obj.get("rationale", "").strip():
            obj["rationale"] = DEFAULT_ACTION["rationale"]
        if obj.get("sample_size", 0) < 1:
            obj["sample_size"] = DEFAULT_ACTION["sample_size"]
        obj["questions"] = []  # must be empty for propose/revise

    return obj


# Sanity checks
sample = '{"action_type": "propose_protocol", "sample_size": 5, "controls": ["control_group"], "technique": "PCR", "duration_days": 3, "required_equipment": [], "required_reagents": [], "questions": [], "rationale": "test", "extra_field": "should_be_stripped"}'
parsed = parse_action(sample)
assert parsed["technique"] == "PCR"
assert "extra_field" not in parsed, "Extra fields should be stripped"
assert parse_action([{"role": "assistant", "content": sample}])["technique"] == "PCR"
assert parse_action([{"role": "user", "content": "some prompt"}, {"role": "assistant", "content": sample}])["technique"] == "PCR"
print("parse_action OK, extra_field stripped:", "extra_field" not in parsed)

parse_action OK, extra_field stripped: True


In [7]:
# ── 6. Reward function ────────────────────────────────────────────────────────
# GRPO calls reward_fn(prompts, completions, **kwargs) and expects a list[float].
# We replay each completion against the environment from the saved seed.
# One-step reward: reset → step(parsed_action) → return reward.

import threading

_client_lock = threading.Lock()
_reward_client = EnvClient(ENV_BASE_URL)


def reward_fn(completions, prompts=None, seed=None, scenario=None, difficulty=None, **kwargs):
    """
    GRPO reward function.

    ``completions``  – list of model-generated strings (one per sample)
    ``seed``         – list of seeds from the dataset column
    ``scenario``     – list of scenario strings from the dataset column
    ``difficulty``   – list of difficulty strings from the dataset column

    Returns a list[float] of rewards, one per completion.
    """
    rewards = []
    n = len(completions)

    for i, completion in enumerate(completions):
        # Each completion in a GRPO batch shares the same prompt index
        idx = i % len(seed) if seed else 0
        ep_seed       = seed[idx]       if seed       else 0
        ep_scenario   = scenario[idx]   if scenario   else "math_reasoning"
        ep_difficulty = difficulty[idx] if difficulty else "easy"

        action = parse_action(completion)
        reward = 0.0

        try:
            with _client_lock:
                _reward_client.reset(
                    seed=ep_seed,
                    scenario=ep_scenario,
                    difficulty=ep_difficulty,
                )
                result = _reward_client.step(action)
            reward = float(result.get("reward", 0.0))

            # Bonus if agreement reached
            if result.get("info", {}).get("agreement_reached"):
                reward += 0.5

            # Small penalty for invalid JSON that fell back to default
            if action == DEFAULT_ACTION:
                reward -= 0.1

        except Exception as e:
            print(f"[reward_fn] error idx={i}: {e}")
            reward = -0.1

        rewards.append(reward)

    return rewards


# Quick sanity check
test_reward = reward_fn(
    completions=[sample],
    seed=[0], scenario=["math_reasoning"], difficulty=["easy"]
)
print("Test reward:", test_reward)

Test reward: [0.25]


In [8]:
# ── 7. Baseline evaluation (before training) ──────────────────────────────────
# Record success rate and average reward on 10 fixed seeds before any training.

def run_greedy_eval(model, tokenizer, seeds, scenario="math_reasoning",
                    difficulty="easy", label="eval"):
    """Run greedy decoding on fixed seeds and return (avg_reward, success_rate)."""
    import torch

    rewards = []
    successes = 0

    model.eval()
    with torch.no_grad():
        for seed in seeds:
            try:
                reset_data = client.reset(seed=seed, scenario=scenario, difficulty=difficulty)
                obs = reset_data["observation"]
                messages = make_chat_messages(obs)

                inputs = tokenizer.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_tensors="pt",
                ).to(model.device)

                out = model.generate(
                    inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,  # greedy
                    temperature=1.0,
                    pad_token_id=tokenizer.eos_token_id,
                )
                gen_ids = out[0][inputs.shape[-1]:]
                text = tokenizer.decode(gen_ids, skip_special_tokens=True)

                action = parse_action(text)
                result = client.step(action)
                r = float(result.get("reward", 0.0))
                if result.get("info", {}).get("agreement_reached"):
                    r += 0.5
                    successes += 1
                rewards.append(r)
            except Exception as e:
                print(f"[eval] seed={seed} error: {e}")
                rewards.append(0.0)

    avg_r = sum(rewards) / max(len(rewards), 1)
    rate  = successes / max(len(seeds), 1)
    print(f"[{label}] avg_reward={avg_r:.4f}  success_rate={rate:.2%}  ({successes}/{len(seeds)} episodes)")
    return avg_r, rate, rewards


print("Loading model for baseline eval...")
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=False,   # must be False for GRPO with Unsloth
)

FastLanguageModel.for_inference(model)
baseline_avg, baseline_rate, baseline_rewards = run_greedy_eval(
    model, tokenizer, EVAL_SEEDS, label="BEFORE training"
)

Loading model for baseline eval...


==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[eval] seed=0 error: string indices must be integers, not 'str'
[eval] seed=1 error: string indices must be integers, not 'str'
[eval] seed=2 error: string indices must be integers, not 'str'
[eval] seed=3 error: string indices must be integers, not 'str'


[eval] seed=4 error: string indices must be integers, not 'str'
[eval] seed=5 error: string indices must be integers, not 'str'
[eval] seed=6 error: string indices must be integers, not 'str'


[eval] seed=7 error: string indices must be integers, not 'str'
[eval] seed=8 error: string indices must be integers, not 'str'
[eval] seed=9 error: string indices must be integers, not 'str'
[BEFORE training] avg_reward=0.0000  success_rate=0.00%  (0/10 episodes)


In [9]:
# ── 8. Attach LoRA adapters for training ──────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

Unsloth: Making `model.base_model.model.model.language_model` require gradients


trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435


In [10]:
# ── 9. GRPO training ──────────────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=GRPO_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.9,
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    bf16=True,
    report_to="none",   # change to "wandb" if you have W&B set up
    # GRPO-specific
    use_vllm=False,     # must be False when using Unsloth
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80 | Num Epochs = 3 | Total steps = 200
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 6,389,760 of 859,375,680 (0.74% trained)


Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


...

[reward_fn] error idx=2: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422


[reward_fn] error idx=3: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422


[reward_fn] error idx=0: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422
[reward_fn] error idx=1: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422


[reward_fn] error idx=2: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422
[reward_fn] error idx=3: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422


[reward_fn] error idx=0: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422
[reward_fn] error idx=1: Client error '422 Unprocessable Entity' for url 'https://ayushozha-replicalab.hf.space/step'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422


In [ ]:
# ── 10. Post-training evaluation ──────────────────────────────────────────────
FastLanguageModel.for_inference(model)

trained_avg, trained_rate, trained_rewards = run_greedy_eval(
    model, tokenizer, EVAL_SEEDS, label="AFTER training"
)

print("\n=== RESULTS ===")
print(f"Baseline  avg reward: {baseline_avg:.4f}  success rate: {baseline_rate:.2%}")
print(f"Trained   avg reward: {trained_avg:.4f}  success rate: {trained_rate:.2%}")
delta_r = trained_avg - baseline_avg
delta_s = trained_rate - baseline_rate
print(f"Delta     avg reward: {delta_r:+.4f}  success rate: {delta_s:+.2%}")

In [ ]:
# ── 11. Reward curve plot ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps   = [e["step"]   for e in log_history if "reward" in e]
rewards = [e["reward"] for e in log_history if "reward" in e]

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, linewidth=2)
plt.xlabel("Step")
plt.ylabel("Mean episode reward")
plt.title("ReplicaLab GRPO — reward curve (Qwen3.5-0.8B)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()
print("Saved reward_curve.png")

In [ ]:
# ── 12. Save model ────────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Optional: push to HuggingFace Hub
if HF_PUSH_REPO:
    model.push_to_hub(HF_PUSH_REPO)
    tokenizer.push_to_hub(HF_PUSH_REPO)
    print(f"Pushed to https://huggingface.co/{HF_PUSH_REPO}")

In [ ]:
# ── 13. Quick interactive demo ────────────────────────────────────────────────
# Run a single episode with the trained model and print each action + reward.
import torch

DEMO_SEED     = 999
DEMO_SCENARIO = "math_reasoning"

reset_data = client.reset(seed=DEMO_SEED, scenario=DEMO_SCENARIO, difficulty="easy")
obs = reset_data["observation"]
print(f"Episode: {reset_data['episode_id']}")
print(f"Paper:   {obs['scientist']['paper_title']}\n")

done = False
total_reward = 0.0

model.eval()
while not done:
    messages = make_chat_messages(obs)
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    action = parse_action(text)
    result = client.step(action)

    rnd = obs['scientist']['round_number'] + 1
    r   = result['reward']
    total_reward += r

    print(f"Round {rnd}: action={action['action_type']}  reward={r:.3f}")
    if action.get('rationale'):
        print(f"         rationale: {action['rationale'][:80]}")

    done = result["done"]
    if not done:
        obs = result["observation"]

print(f"\nEpisode done. Total reward: {total_reward:.3f}")
print("Agreement reached:", result.get("info", {}).get("agreement_reached"))